<a href="https://colab.research.google.com/github/DavidSenseman/STA1403/blob/main/Chapter_11A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- STA1403_CHAPTER_11A:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **STA 1403: Biostatistics**

## **Chapter 11 : Regression Analysis**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

#### Contents
* **11.1 : Introduction**
* **11.2 : Linear Regression Models with One Binary Explanatory
Variable**
* **11.3 : Statistical Inference Using Simple Linear Regression
Models**
* **11.4 : Linear Regression Models with One Numerical**
Explanatory Variable
* 11.5 : Goodness of Fit
* 11.6 : Model Assumptions and Diagnostics
* 11.7 : Multiple Linear Regression
* 11.8 : Advanced

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which **_is_** your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.139.115.187",
  "hostname": "187.115.139.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## Test Your STA1403 Key

Run the next cell to test whether your STA1403 Key is correctly installed in your Colab Secrets.

In [ ]:
# @title Test Your STA1403 KEY

from google.colab import userdata
import os

# Check if STA1403 key is properly loaded
try:
    # 1. Get the key from Secrets
    STA1403_KEY = userdata.get('STA1403_KEY')

    # 2. Set it as an environment variable
    os.environ['STA1403_KEY'] = STA1403_KEY

    print("API key loaded and environment variable set successfully!")
    print(f"Key length: {len(STA1403_KEY)}")

except Exception as e:
    print(f"Error loading STA1403 key: {e}")
    print("Please set your STA1403 key in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'STA1403_KEY'")
    print("3. Paste your STA1403 key and toggle 'Notebook access' on")

If your STA1403 key is correctly stored in your Colab Secrets, you should see the following output:
```text
STA1403 key loaded and environment variable set successfully!
Key length: 28
```
Contact your Instructor if you recieve an error since you won't be able to submit your lesson for grading unless your STA1403 key is correctly installed.

## Load Data Sets

Run the next cell to load the data sets for this lesson.

In [ ]:
# @title Load Data Sets
import pandas as pd

# Load dataset
saltBP_df = pd.read_csv("https://biologicslab.co/STA1403/data/saltBP.txt",
                         sep = ' ', na_values=[" ", "NA", "null", ""])

# Load dataset
bodyTemp_df = pd.read_csv("https://biologicslab.co/STA1403/data/BodyTemperature.txt",
                         sep = ' ', na_values=[" ", "NA", "null", ""])

print("Data sets loaded sucessfully. ✅")

# **Chapter 11 :  Regression Analysis**

## **11.1 Introduction**

In Chap. 8, we discussed testing a hypothesis regarding the relationship between a binary explanatory variable (referred to as the factor) and a numerical response variable using two-sample t-tests. We also discussed situations where we investigate the linear relationship between two numerical variables (e.g., percent body fat and abdomen circumference). In this case, we typically consider one of the two numerical variables (e.g., percent body fat) as the response (or target) variable and the other one (e.g., abdomen circumference) as the explanatory variable. The common theme for both methods is that we investigate the relationship between an explanatory variable (either categorical or numerical) and a numerical random variable. In this chapter, we discuss an alternative approach for analyzing such problems. This approach uses **linear regression models** for either testing a hypothesis regarding the relationship between one or more _explanatory variables_ and a response variable, or **predicting** unknown values of the response variable using one or more _predictors_. Note that we use the terms "explanatory variables" and "predictors" to distinguish the role of variables (other than the response variable) in the model.

---------------

>Occasionally, we might want to use linear regression models for both hypothesis testing and prediction. However, in most cases, our objective is either examining the relationship between the response variable and a set of explanatory variables, or predicting the unknown values of the response variable using a set of predictors. It is very important to specify our objective clearly before starting the analysis. As we discuss later in this chapter, our strategy to build a linear regression model depends on our objective.
------------

Throughout this chapter, we use $X$ to denote explanatory variables and $Y$ to denote response variables. We start by focusing on problems where the explanatory variable is binary. As before, the binary variable $X$ can be either $0$ or $1$. We then continue our discussion for situations where the explanatory variable is numerical. Finally, we discuss linear regression problems where there are more than one explanatory variable (possibly, a combination of binary and numerical variables).

## **11.2 Linear Regression Models with One Binary Explanatory Variable**

Suppose that we want to investigate the relationship between sodium chloride (salt) consumption and blood pressure among elderly people (e.g., above $65$ years old). We take a random sample of $25$ people from this population and find that $15$ of them keep a low sodium chloride diet (less than $6$ grams of salt per day) and $10$ of them keep a high sodium chloride diet (more than $6$ grams of salt per day). We use the variable $X$ for the sodium chloride intake level, where $X = 0$ means low sodium chloride intake (group $1$), and $X = 1$ means high sodium chloride intake (group $2$). For people in our sample, we measure systolic blood pressure, which we denote as $Y$. Therefore, for each individual $i$ in our sample, we have a pair of observations $(x_{i}, y_{i})$, where the first element shows the group (low or high sodium chloride diet), and the second element shows the blood pressure measure. Figure 11.1 shows the dot plot for the observed data. As we can see, blood pressure tends to be higher among people with high sodium chloride diet.

![__](https://biologicslab.co/STA1403/images/chapter_11/chapter_11A_image02A.png)

>**Fig. 11.1** The dot plot for systolic blood pressure for $25$ elderly people, where $15$ people follow a low sodium chloride diet ($X = 0$), and $10$ people follow a high sodium chloride diet ($X = 1$)

Figure 11.1 shows the dot plot for the observed data. As we can see, blood pressure tends to be higher among people with high sodium chloride diet. Figure 11.2 shows the dot plot along with sample means, shown as black circles,for each group. In this graph, the difference between the two sample means is denoted as b. Recall that the difference between the sample means was what we used to perform the two-sample t-test. By connecting the two sample means, we can show the overall pattern for how blood pressure changes from one group to another.

![__](https://biologicslab.co/STA1403/images/chapter_11/chapter_11A_image03A.png)

>**Fig. 11.2** The dot plot for systolic blood pressure for 25 elderly people. Here, the sample means among the low and high sodium chloride diet groups are shown as _black circles_. A _straight line_ connects the sample means. The line intercepts the vertical axis at $a = 133.17$ and has slope $b = 6.25$.

The sample mean among the first group (the black point in the left) is regarded as our point estimate for population mean of systolic blood pressure among people with low sodium chloride diet $(X = 0)$. If we do not know someone's blood pressure measure, y, but we know that she belongs to the first group (i.e., $x = 0$), the sample mean provides a reasonable point estimate of her blood pressure. We show this as $\hat{y}{x=0}$ and interpret it as the estimate of the response variable among individuals whose value of the explanatory variable is x = 0. Similarly, we can use the sample mean for the second group (the black point in the right) as our point estimate of the response variable (i.e., blood pressure) among people whose value of the explanatory variable (sodium chloride intake level) is $x = 1$. We denote this estimate as $\hat{y}{x=1}$.

For the above example, the sample means of blood pressure for the first and second groups are $133.17$ and $139.42$, respectively. Therefore, $\hat{y}{x=0}=133.17$ and $\hat{y}{x=1}=139.42$.

Now consider the straight line that connects the two point estimates $ \hat{y}{x=0} $ and $ \hat{y}{x=1}$. This line contains both point estimates (black points); hence, for any value of $x$ (i.e., whether $x = 0$ or $x = 1$), the line can be used to find the estimate of the response variable (blood pressure). We denote this estimate as $ \hat{y}$. Because the horizontal difference between the two points is $1$ unit, and the vertical difference between the two points is $b = \hat{y}_{x=1}-\hat{y}_{x=0}$, the slope of this line is $b/1 = b$. The line intercepts the vertical axis at $a$. In this case, $a = \hat{y}_{x=0}$. For the above example, $a = 133.17$ and $b = 139.42 - 133.17 = 6.25$.

----------

>Using the intercept $a$ and slope $b$, we can write the equation for the straight line (Fig. 11.2) that connects the estimates of the response variable for different values of $X$ as follows:
$$\hat{y} = a + b x. \tag{11.2}
$$
The above equation specifies a straight line called the **regression line**. The regression line captures the linear relationship between the response variable (here, blood pressure) and the explanatory variable (here, "low" versus "high" sodium chloride diet).
The slope of the regression line has a central role in capturing the linear relationship between the response variable and the explanatory variable: the slope $b$ is interpreted as our estimate of the expected (average) change in the response variable associated with one unit increase in the value of the explanatory variable. Note that in general, we _cannot interpret_ this as the amount of increase in the response variable caused by one unit increase in the explanatory variable unless the data are obtained through a randomized experiment, where the value of the explanatory variable is changed by intervention.

-----

For the blood pressure example, the regression line is
$$
\hat {y} = 1 3 3. 1 7 + 6. 2 5 x.
$$
Based on the slope of $b=6.25$, we expect that on average the blood pressure increases by $6.25$ units for one unit increase in the value of the explanatory variable. In this case, one unit increase in $X$ from $0$ to $1$ means moving from low sodium chloride diet group to high sodium chloride diet group.

We can use the regression line to estimate the blood pressure of a subject given his or her sodium chloride intake level. For an individual with $x=0$ (i.e., low sodium chloride diet), the estimate according to the above regression line is
$$
\begin{array}{l} \hat{y} = a + b \times 0 = a \\
= \hat {y} _ {x = 0}, \\
\end{array}
$$
which is the sample mean for the first group. For an individual with $x=1$ (i.e., high sodium chloride diet), the estimate according to the above regression line is
$$
\begin{array}{l}
\hat{y} = a + b \times 1 = a + b \\
= \hat {y}_{x = 0} + \hat {y}_{x = 1} - \hat {y}_{x = 0} \\
= \hat {y} _ {x = 1}. \\
\end{array}
$$
Note that in this case, where the explanatory variable is binary with $0–1$ values, we can evaluate the equation for the regression line at $x=0$ and $x=1$ only.

Now consider the observed data for our sample of 25 people. For each individual in this sample, there is a difference between the actual observed blood pressure and the estimate we obtain from the regression line. For individual $i$, whose values of the explanatory variable and the response variable are $x_{i}$ and $y_{i}$, respectively, the estimated value of the response variable, denoted as $\hat{y}_{i}$, is
$$
\hat {y}_{i} = a + b x_{i}.
$$

----------

>We refer to the difference between the observed and estimated values of the response variable as the **residual**. For individual $i$, we denote the residual $e_{i}$ and calculate it as follows:
$$
e_{i} = y_{i} - \hat {y}_{i}.
$$
By rearranging the terms in the above equation, we can write the observed value $y_{i}$ in terms of the estimate obtained from the regression line and the corresponding residual,
$$ \begin{array}{l}
y_{i} = \hat {y}_{i} + e_{i} \\
= a + bx_{i} + e_{i}. \\
\end{array}
$$

--------------

For the blood pressure example, $\hat{y}=a$ for all individuals in the first group, and $\hat{y}=a+b$ for all individuals in the second group. For instance, if an individual $i$ belongs to the first group, $x_{i}=0$, her estimated blood pressure is $\hat{y}{i}=a=133.17$. Now if the observed value of her blood pressure is $y{i}=135.08$, then the residual is
$$
e_{i} = 135.08 - 133.17 = 1.91.
$$
In Fig. 11.2, the residuals $ e_{i} $ and $ e_{j} $ are shown as vertical arrows for two individuals (one from each group). For individual i (in the first group), the residual is positive since $y_{i}$ is greater than $\hat{y}{i}=a$. For individual $j$ (in the second group), the residual is negative since $y{j}$ is less than $ \hat{y}_{j}=a+b $ . The directions of the arrows show the sign of the residuals: upward for positive residuals and downward for negative residuals.

When the explanatory variable is binary, the residuals are in fact deviations from the corresponding group sample means. As mentioned in Chap. 2, the sum of the deviations from the sample mean over all observed values is always zero. Therefore, the sum (hence, the mean) of all the residuals is zero. The sum of the squares of the residuals, however, is not zero in general (it is a positive value) and is commonly used as a measure of overall discrepancy between observed values and estimates from the regression line. This is analogous to the within-group sum of squares measure, $SS_{W}$, we used for ANOVA.

---------------

>As a measure of discrepancy between the observed values and those estimated by the line, we calculate the **Residual Sum of Squares** (RSS):
$$
RSS = \sum_{i}^{n} e_{i}^{2}. \tag {11.1}
$$
Here, $e_{i}$ is the residual of the $i$th observation, and $n$ is the sample size. The square of each residual is used so that its sign (i.e., the direction of the arrows in Fig. 11.2) becomes irrelevant.

-------------

To capture the overall change in blood pressure from one group to another, we decided to draw a line by connecting the sample means. We could have of course chosen different lines between the two groups; for example, we could have connected the sample medians. For any possible line, we can define the residuals as before (i.e., the vertical difference between the observed values and the line) and calculate RSS. It turns out that among all possible straight lines we could have drawn, the linear regression line discussed above provides the smallest value of RSS. Therefore, the above approach for finding the regression line is called the **least-squares** method, and the resulting line is called the **least-squares regression line**.

### **_Using Python for Finding Regression Lines_**

For the blood pressure example, we can simply find the regression line by calculating $a$ and $b$ using the sample means of blood pressure for the two groups separately. However, Python can use the function `ols()` to find $a$ and $b$ directly without calculating the sample means as shown in Example 1.

## Example 1: Find Regression Line

The code in the cell below finds the regression line for the blood pressure example (`saltBP_df`) using the function `ols()` from `statsmodels.api`. This is actually the same function that we used previously to compute an Analysis of Variance (ANOVA).

In [ ]:
# @title Example 1: Find Regression Line

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# ── 1. Variable definitions ──────────────────────────────────────────────────────
dependent_var = "BP"        # Response variable
predictor     = "saltLevel" # Predictor
dataset       = saltBP_df   # DataFrame

# ── 2. Simple linear regression ──────────────────────────────────────────────────
formula = f"{dependent_var} ~ {predictor}"
model = ols(formula, data=dataset).fit()
print(f"ols formula: {formula} \n")

# ── 3. Significance stars helper ─────────────────────────────────────────────────
def get_sig_stars(p):
    if pd.isna(p):   return ""
    if p < 0.001:    return "***"
    elif p < 0.01:   return "**"
    elif p < 0.05:   return "*"
    elif p < 0.1:    return "."
    else:            return ""

# ── 4. Format p-value the R way (<2e-16 for tiny values) ────────────────────────
def fmt_p(p):
    if p < 2e-16:
        return "< 2e-16"
    elif p < 0.001:
        return f"{p:.2e}"
    else:
        return f"{p:.7f}".rstrip("0")   # trim trailing zeros like R does

# ── 5. Build and print the Coefficients table ─────────────────────────────────
params   = model.params
bse      = model.bse
tvalues  = model.tvalues
pvalues  = model.pvalues

print("Coefficients:")
header = f"{'':15s} {'Estimate':>10s} {'Std. Error':>10s} {'t value':>8s} {'Pr(>|t|)':>12s}"
print(header)

for name in params.index:
    p     = pvalues[name]
    stars = get_sig_stars(p)
    display_name = f"({name})" if name == "Intercept" else name
    row = (
        f"{display_name:15s} "
        f"{params[name]:>10.3f} "
        f"{bse[name]:>10.3f} "
        f"{tvalues[name]:>8.3f} "
        f"{fmt_p(p):>12s} "
        f"{stars}"
    )
    print(row)

print("---")
print("Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1")

# ── 6. Footer stats ───────────────────────────────────────────────────────────
rse      = np.sqrt(model.mse_resid)
df_resid = int(model.df_resid)
r2       = model.rsquared
r2_adj   = model.rsquared_adj
f_stat   = model.fvalue
f_df1    = int(model.df_model)
f_df2    = int(model.df_resid)
f_pval   = model.f_pvalue

print(f"\nResidual standard error: {rse:.3f} on {df_resid} degrees of freedom")
print(f"Multiple R-squared:  {r2:.4f},\tAdjusted R-squared:  {r2_adj:.4f}")
print(f"F-statistic: {f_stat:.2f} on {f_df1} and {f_df2} DF,  p-value: {fmt_p(f_pval)}")

If the code is correct, you should see the following output:
```text
ols formula: BP ~ saltLevel

Coefficients:
                  Estimate Std. Error  t value     Pr(>|t|)
(Intercept)        133.173      1.007  132.222      < 2e-16 ***
saltLevel            6.256      1.593    3.929     6.72e-04 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 3.901 on 23 degrees of freedom
Multiple R-squared:  0.4016,	Adjusted R-squared:  0.3756
F-statistic: 15.43 on 1 and 23 DF,  p-value: 6.72e-04

```

The output above provides a table with the title `Coefficients`, where the first column, called `Estimate`, includes the intercept $133.17$ and the slope $6.25$. The output, of course, includes more information regarding statistical inference using regression analysis, which is the focus of the next section.

## **Exercise 1: Find Regression Line**

In the cell below, use the “BodyTemperature.txt” data set to build a simple linear regression model for body temperature using heart rate as the predictor.

**Code Hints:**

Change the variable definitions to read as follows:
```Python
# ── 1. Variable definitions ──────────────────────────────────────────────────────
dependent_var = "Temperature"  # Response variable
predictor     = "HeartRate"    # Predictor
dataset       = bodyTemp_df    # DataFrame
```

In [ ]:
# @title Exercise 1: Find Regression Line



If the code is correct, you should see the following output:
```text
ols formula: Temperature ~ HeartRate

Coefficients:
                  Estimate Std. Error  t value     Pr(>|t|)
(Intercept)         92.391      1.201   76.900      < 2e-16 ***
HeartRate            0.081      0.016    4.956     3.01e-06 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.860 on 98 degrees of freedom
Multiple R-squared:  0.2004,	Adjusted R-squared:  0.1923
F-statistic: 24.56 on 1 and 98 DF,  p-value: 3.01e-06
```

This output represents a **Simple Linear Regression** analysis modeling the relationship between **HeartRate** (independent variable) and **Temperature** (dependent variable).

#### **The Regression Equation**
Based on the "Estimate" column, the mathematical model is:
$$\text{Temperature} = 92.391 + 0.081 \times (\text{HeartRate})$$

*   **Intercept (92.391):** The predicted Temperature when HeartRate is zero.
*   **Slope (0.081):** For every **1 unit increase in HeartRate**, the **Temperature is expected to increase by 0.081 units**.

#### **Statistical Significance**
*   **P-value (3.01e-06):** The p-value for `HeartRate` is extremely small (well below the standard 0.05 threshold). This indicates that the relationship is **statistically significant**.
*   **t-value (4.956):** A high t-value confirms that the HeartRate coefficient is significantly different from zero.
*   **Significance Codes (***):** The triple asterisks confirm the highest level of statistical significance.

#### **Goodness of Fit**
*   **Multiple R-squared (0.2004):** This indicates that **20.04% of the variance in Temperature can be explained by HeartRate**.
    *   *Interpretation:* While the relationship is significant, HeartRate only explains a small portion of the total variance; other factors are likely influencing Temperature as well.
*   **Residual Standard Error (0.860):** On average, the model's predictions deviate from the actual observed temperatures by approximately 0.86 units.
*   **F-statistic (24.56):** The very low p-value (3.01e-06) for the F-statistic confirms that the model as a whole is statistically significant.

## **11.3 Statistical Inference Using Simple Linear Regression Models**

In the previous section, we discussed finding regression lines with binary explanatory variable. We showed how we can find the intercept and slope of the regression line and discussed how we can use regression lines to estimate the values of the response variable. As usual, we would like to extend our findings to the entire population. This is the topic of this section.

Using the regression line, we can estimate the unknown value of the response variable for members of the population who did not participate in our study. In this case, we refer to our estimates as **predictions**. For example, we can use the linear regression model we built in the previous section to predict the value of blood pressure for a person with high sodium chloride diet (i.e., x = 1),
$$
\begin{array}{l}
\hat{y} = 133.17 + 6.25 x \\
= 133.17 + 6.25 \times 1 \\
= 139.42. \\
\end{array}
$$
Next, we want to use the linear regression model to comment on the type and strength of the relationship between $Y$ and $X$. Using the observed data, the regression line captures the linear relationship between the response variable and the explanatory variable as follows:

$$\hat{y} = a + b x.$$

Based on this line, we can write the value of the response variable for individual $i$ in terms of the above regression line and the residual:
$$
y_{i} = a + b x_{i} + e_{i}.
$$

------------

>The linear relationship between $Y$ and $X$ in the entire population can be presented in a similar form,
$$ Y = \alpha + \beta X + \varepsilon , $$
(11.2)
where $\alpha$ is the intercept, and $\beta$ is the slope of the regression line if we had used the entire population to find the regression line. Here, $ \varepsilon $ is called the **error term**, representing the difference between the estimated (based on the regression line for the entire population) and the actual values of Y in the population. We refer to the above equation as the **linear regression model**. More specifically, we call it the **simple linear regression model** since there is only one explanatory variable. We refer to $\alpha$ and $\beta$ as the **regression parameters**. More specifically, $ \beta $ is called the **regression coefficient** for the explanatory variable. The process of finding the regression parameters is called **fitting** a regression model to the data.

-------------------

As usual, the regression parameters for the population remain unknown. We estimate these parameters using a random sample from the population. Suppose that we have a sample of size $n$: $(X_{1}, Y_{1})$, $(X_{2}, Y_{2})$, $\ldots$, $(X_{n}, Y_{n})$. In this case, we have a pair of values (one for the explanatory variable and one for the response variable) for each individual in our sample. Using this sample, we can estimate the regression parameters by fitting a linear regression model to the observed data as described above:
$$ \hat {Y} = A + B X. $$

Here, $A$ and $B$ are statistics (i.e., calculated based on the observed data only), which are used as estimators. We used capital letters since A and B themselves are random variables and their values can change every time we take a new sample of size n from the population.

Of course, as before, we only have one such sample, $(x_{1}, y_{1})$, $(x_{2}, y_{2})$, $ \ldots $, $(x_{n}, y_{n})$. Using the observed data, we find the point estimates $a$ and $b$ (i.e., the specific values of estimators $A$ and $B$) for $\alpha$ and $\beta$, respectively, following the steps discussed in the previous chapter. For blood pressure example, the point estimates for $\alpha$ and $\beta$ were $a = 133.17$ and $b = 6.25$. These estimates are provided in the first column of the `Coefficients` table under `Estimate` in the output from Example 1.

As discussed in earlier chapters, the point estimates do not reflect our uncertainty; we always remain uncertain about the actual values of $\alpha$ and $\beta$, and our estimates for these parameters can change when we take a different sample from the population. Therefore, we use the point estimates a and b obtained from the observed data along with the sampling distribution of the estimators A and B to find confidence interval estimates for $\alpha$ and $\beta$ . This is similar to what we had for the population mean. Because the slope parameter $ \beta$ has a central role in capturing the linear relationship between $Y$ and $X$, we focus on finding its confidence intervals. (Similar approach can be used for the intercept $\alpha$.)

### **_11.3.1 Confidence Interval for Regression Coefficients_**

Finding confidence intervals for $\beta$ is quite similar to the approach we used to find confidence intervals for the population mean. First, we need to find the standard error (i.e., estimated standard deviation of the sampling distribution of $B$) of our estimate. We denote this as $SE_{b}$. Next, we need to specify the required confidence level, c, and find its corresponding factor, $t_{crit}$. Then, we can find the confidence interval for $ \beta $ as follows:
$$
[b - t_{\mathrm{crit}} \times SE_{b}, b + t_{\mathrm {crit}} \times SE_{b} ].
$$

For simple linear regression models, $ SE_{b} $ is obtained as follows:
$$
SE_{b} = \frac {\sqrt {R S S / (n - 2)}}{\sqrt {\sum_{i} \left(x_{i} - \bar {x}\right)^{2}}}, \tag{11.3}
$$
where $x_{i}$ are the observed values of the explanatory variable, which takes either $0$ or $1$, and $\bar{x}$ is the sample mean (which is the same as the sample proportion for a binary variable). When we fit a linear regression model using Python, it provides the standard error of the regression coefficient. From the `Coefficients` table in the output from Example 1, the `Std. Error` column provides the standard error for intercept as $SE_{a}=1.01$ and the standard error for the regression coefficient as $SE_{b}=1.59$.

The steps to find $t_{crit}$ are the same as the steps discussed in Chap. 6 for the population mean. Here, however, $t_{crit}$ is obtained from the $t$-distribution with $n - 2$ degrees of freedom. In our example, the sample size is $n = 25$. Therefore, we use the $t$-distribution with $25 - 2 = 23$ degrees of freedom. If we set the confidence level to $0.95$, then $t_{crit} = 2.07$, which is obtained from the $t$-distribution with $23$ degrees of freedom by setting the upper tail probability to $(1 - 0.95)/2 = 0.025$.

Therefore, the 95% confidence interval for $ \beta $ is
$$
[6.25 - 2.07 \times 1.59, 6.25 + 2.07 \times 1.59 ] = [2.96, 9.55].
$$

At $0.95$ confidence level, the slope of the regression line is between $2.96$ and $9.55$. In other words, by moving from the low sodium chloride diet to high sodium chloride diet, the expected (average) amount of increase in blood pressure is estimated to be somewhere between $2.96$ and $9.55$ units.

Alternatively, we can use Python to obtain the confidence intervals. As our estimates for $\alpha$ and $\beta$ change, the least-squares regression line changes. Therefore, we can obtain confidence intervals for the regression line and predictions we obtain based on this line.

Example 3 showw how to obtain the 95% confidence interval of the regression line for the blood pressure example.

## Demonstration 1: 95% Confidence Interval for Regression Line

The code in the cell below, generate the 95% confidence interval for the regression line for the relationship between salt intake and blood pressure using the data in the `saltBP_df` DataFrame.

In [ ]:
# @title Demostration 1: 95% Confidence Interval for Regression Line

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from statsmodels.formula.api import ols

# ── 1. Variable definitions ──────────────────────────────────────────────────────
dependent_var = "BP"          # Response variable
predictor     = "saltLevel"   # Predictor
dataset       = saltBP_df     # DataFrame

# ── 2. Simple linear regression: BP ~ saltLevel ───────────────────────────────
formula = f"{dependent_var} ~ {predictor}"
model = ols(formula, data=dataset).fit()

# ── 3. Prediction grid with 95% confidence interval ──────────────────────────
x_grid = np.linspace(0, 1, 200)
x_pred = sm.add_constant(pd.DataFrame({predictor: x_grid}))
predictions = model.get_prediction(x_pred)
pred_summary = predictions.summary_frame(alpha=0.05)  # 95% CI

y_fit  = pred_summary["mean"]
y_lower = pred_summary["mean_ci_lower"]
y_upper = pred_summary["mean_ci_upper"]

# Set plot size
fig, ax = plt.subplots(figsize=(6, 5))

# ── 4. Regression line (solid black) ─────────────────────────────────────────
ax.plot(x_grid, y_fit, color="black", linewidth=2.0)

# ── 5. Confidence interval bands (dashed red) ────────────────────────────────
ax.plot(x_grid, y_lower, color="red", linewidth=1.2, linestyle="--")
ax.plot(x_grid, y_upper, color="red", linewidth=1.2, linestyle="--")

# ── 6. Rug plot on x-axis (tick marks showing observed saltLevel values) ──────
rug_vals = dataset[predictor].values
for xv in rug_vals:
    ax.plot([xv, xv], [ax.get_ylim()[0], ax.get_ylim()[0] + 0.15],
            color="black", linewidth=0.8, clip_on=False,
            transform=ax.get_xaxis_transform())

# ── 7. Title, axis labels ─────────────────────────────────────────────────────
ax.set_title(f"{predictor} effect plot", fontsize=13, fontweight="medium", pad=10)
ax.set_xlabel(predictor, fontsize=12)
ax.set_ylabel(dependent_var, fontsize=12)

# ── 8. Axis limits and ticks ──────────────────────────────────────────────────
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(130, 142)
ax.xaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax.yaxis.set_major_locator(ticker.MultipleLocator(2))

# ── 9. R-style frame with ticks on all four sides ────────────────────────────
ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
ax.tick_params(axis="both", direction="in", which="both",
               top=True, right=True, labelsize=11)

plt.tight_layout()
plt.show()

If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_11/chapter_11A_image10A.png)

The output shows the least-squares regression line (solid line) and its 95% confidence interval (dashed curves) for the relationship between blood pressure and sodium chloride intake level. In this plot, the solid straight line is the least-squares regression line for the observed data, and the dashed curves show the 95% confidence interval for the regression line. The curves show how much the regression line can change as we take different samples from the population.

### **_11.3.2 Hypothesis Testing with Simple Linear Regression Models_**

Linear regression models can be used for testing hypotheses regarding possible linear relationship between the response variable and the explanatory variable. For this, the regression coefficient $\beta$, its estimator $B$, and its point estimate $b$ play a central role. For linear regression models with a binary explanatory variable, we found the regression line by connecting the sample means of the two groups. In these models, the slope $b$ is the difference between the two sample means. Similarly, the regression coefficient $\beta$ captures the difference between the population means for the two groups. If the response variable is not related to the binary explanatory variable, the two population means are the same, and the slope of the regression line for the whole population will be zero. That is, the null hypothesis stating no relationship between the two variable can be written as $H_{0}:\beta=0 $.

This is analogous to the application of the two-sample $t$-test for hypothesis testing regarding the relationship between a numerical variable and a binary variable. Similar to the two-sample $t$-test, we need to obtain the $t$-score as the observed value of the test statistic. Recall that we obtained the $t$-score by dividing the observed difference between the sample means by its standard error. Similarly, we find the $t$-score for linear regression models as follows:
$$ t = \frac {b}{SE_{b}}. $$
Then, we find the $p$-value (i.e., the observed significance level) by calculating the probability of as or more extreme values than $t$-score under the null hypothesis.

-------------

>To assess the null hypothesis $H_{0}:\beta=0$, which is interpreted as no linear relationship between the response variable and the explanatory variable, we first calculate the $t=b/SE_{b}$ and find the corresponding $p$-value as follows:
$$
\begin{array}{l}
\mathrm {if } H_{A}: \beta < 0, \quad p_{\mathrm {obs}} = P(T \leq t), \\
\mathrm {if } H_{A}: \beta > 0, \quad p_{\mathrm {obs}} = P(T \geq t), \\
\mathrm {if } H_{A}: \beta \neq 0, \quad p_{\mathrm {obs}} = 2 \times P(T \geq | t |), \\
\end{array}
$$
where $T$ has the $t$-distribution with $n-2$ degrees of freedom.

-----------

In the blood pressure example, the estimate of the regression coefficient was $b=6.25$, and the standard error was $SE_{b}=1.59$. Therefore,
$$ t = \frac {6 .25}{1.59} = 3.93. $$
If $H_{A}:\beta\neq0$ (which is the common form of the alternative hypothesis), we find the $p$-value by calculating the upper tail probability of $|3.93|=3.93$ from the $t$-distribution with $25-2=23$ degrees of freedom and multiplying the result by 2. For this example, $p_{\mathrm{obs}}=2\times0.00033=0.00066$.

Because $p_{obs}$ for this example is quite small and below any commonly used confidence level (e.g., $0.01$, $0.05$, $0.1$), we can reject the null hypothesis and conclude that blood pressure is related to sodium chloride diet level.

When we use Python to fit a linear regression model, the output provides the $t$-score and its corresponding $p$-value. For the output from Example 1, the column with the title $t$ value provides the $t$-scores for $\alpha$ and $\beta$, and the last column, $\Pr(>|t|)$, provides the corresponding $p$-values.

## **11.4 Linear Regression Models with One Numerical Explanatory Variable**

In the previous section, to study the relationship between blood pressure and daily salt intake, we used a binary variable that indicates whether the amount of daily sodium chloride intake for each individual is above a certain cutoff (6 grams per day). The data would be more informative of course if we use the actual amount of sodium chloride intake per day. By doing so, the explanatory variable becomes numerical (quantitative) as opposed to binary.

In this section, we discuss simple linear regression models (i.e., linear regression with only one explanatory variable), where the explanatory variable is numerical. For the most part, we use the same concepts for explaining the model, follow the same steps to fit the model, and interpret the output of the model same way. Here, of course, the explanatory variable can take more than two values. In fact, in many cases (e.g., sodium chloride intake per day), it can take an uncountable number of possible values.

We start our analysis by creating the scatter plot of the response variable and the explanatory variable. Example 4 shows how to do this.

## Example 2: Generate Scatterplot

The code in the cell below generates a scatterplot showing body temperature (`Temperature`) as the predictor ($x$) and heart rate (`HeartRate`) as the dependent variable ($y$).

In [ ]:
# @title Example 2: Generate Scatterplot

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── 1. Variable definitions ──────────────────────────────────────────────────────
dependent_var = "HeartRate"          # Response variable
predictor     = "Temperature"        # Predictor
dataset       = bodyTemp_df          # DataFrame

# Set plotting environment
fig, ax = plt.subplots(figsize=(6, 5))

# Scatter plot with open circles
ax.scatter(dataset[predictor], dataset[dependent_var],
           facecolors="none", edgecolors="black",
           s=60, linewidths=0.8)

# Axis labels
ax.set_xlabel(predictor, fontsize=13)
ax.set_ylabel(dependent_var, fontsize=13, rotation=90, labelpad=20)

# ── 8. Axis limits and ticks ──────────────────────────────────────────────────
x_min, x_max = dataset[predictor].min(), dataset[predictor].max()
y_min, y_max = dataset[dependent_var].min(), dataset[dependent_var].max()

padding_x = (x_max - x_min) * 0.1
padding_y = (y_max - y_min) * 0.1

ax.set_xlim(x_min - padding_x, x_max + padding_x)
ax.set_ylim(y_min - padding_y, y_max + padding_y)

# axis ranges
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax.yaxis.set_major_locator(ticker.MultipleLocator(5))

# Box around the plot, ticks on bottom and left only
ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
ax.tick_params(axis="both", direction="out", which="both",
               top=False, right=False, labelsize=11)

plt.tight_layout()
plt.show()

If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_11/chapter_11A_image12A.png)

## **Exercise 2: Generate Scatterplot**

In the cell below, write the code to generate a scatterplot with `salt` as the predictor ($x$) and `BP` as the dependent variable ($y$).

**Code Hints:**

1. Copy-and-paste Example 2 into the cell below.
2. Change the variable definitions to read as follows:
```Python
# ── 1. Variable definitions ──────────────────────────────────────────────────────
dependent_var = "BP"        # Response variable
predictor     = "salt"      # Predictor
dataset       = saltBP_df   # DataFrame
```

In [ ]:
# @title Exercise 2: Generate Scatterplot



If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_11/chapter_11A_image11A.png)

The output shows a scatterplot of blood pressure by sodium chloride intake. As we can see, there is a clear upward trend indicating that increase in sodium chloride intake tends to coincide with increase in blood pressure. Moreover, the trend seems to be linear, so a straight line can capture the overall pattern. Indeed, the process of fitting a linear regression model to the data involves finding a straight line that can be considered as the best representation of the overall relationship between blood pressure and sodium chloride intake.

The left panel of Fig. 11.7 shows the scatterplot of blood pressure by daily sodium chloride intake along with some candidate lines for capturing the overall relationship between the two variables. To choose a line, we need to explain what we mean by the “best representation” of the data. Similar to the approach we used earlier in this chapter, we measure the discrepancy between each line and the observed data in terms of the residual sum of squares ($RSS$), and choose the line with the smallest value of $RSS$.

![__](https://biologicslab.co/STA1403/images/chapter_11/chapter_11A_image06A.png)

>**Fig. 11.7** _Left panel:_ Scatterplot of blood pressure by daily sodium chloride intake along with some candidate lines for capturing the overall relationship between the two variables. The black line is the least-squares regression line. _Right panel:_ The least-squares regression line for the relationship between blood pressure and sodium chloride intake. The vertical arrows show the residuals for two observations. The stars are the estimated blood pressure for daily sodium chloride intakes from 0 to 14 grams.

The code in the demostration in the next cell shows the Python code to recreate the _Right panel_ of Fig. 11.7.

## **Demonstration 2: Recreate Fig. 11.7 (Right Panel)**

The code in the cell below shows how to recreate the right panel of **Fig. 11.7**.

In [ ]:
# @title Demonstration 2: Recreate Fig. 11.7 (Right Panel)

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from statsmodels.formula.api import ols

fig, ax = plt.subplots(figsize=(6, 5))

# ── 1. Fit the regression model ───────────────────────────────────────────────
model = ols("BP ~ salt", data=saltBP_df).fit()
a = model.params["Intercept"]
b = model.params["salt"]

# ── 2. Regression line ────────────────────────────────────────────────────────
x_line = np.linspace(0, 14, 200)
y_line = a + b * x_line
ax.plot(x_line, y_line, color="black", linewidth=1.5)

# ── 3. Stars (*) at predicted values for salt = 0, 1, 2, ..., 14 ─────────────
x_stars = np.arange(0, 15, 1)
y_stars = a + b * x_stars
ax.scatter(x_stars, y_stars, marker="*", color="black", s=80, zorder=5)

# ── 4. Observed data points (open circles) ────────────────────────────────────
ax.scatter(saltBP_df["salt"], saltBP_df["BP"],
           facecolors="none", edgecolors="black",
           s=60, linewidths=0.8, zorder=4)

# ── 5. Pick residual examples: one below (i) and one above (j) the line ───────
saltBP_df = saltBP_df.copy()
saltBP_df["fitted"] = model.fittedvalues
saltBP_df["resid"]  = model.resid

# Point i: largest negative residual in salt ~3-5 range
mask_i = (saltBP_df["salt"] > 3) & (saltBP_df["salt"] < 5)
idx_i  = saltBP_df.loc[mask_i, "resid"].idxmin()
xi, yi, yhat_i = saltBP_df.loc[idx_i, ["salt", "BP", "fitted"]]

# Point j: largest positive residual in salt ~10-12 range
mask_j = (saltBP_df["salt"] > 10) & (saltBP_df["salt"] < 12)
idx_j  = saltBP_df.loc[mask_j, "resid"].idxmax()
xj, yj, yhat_j = saltBP_df.loc[idx_j, ["salt", "BP", "fitted"]]

# ── 6. Dashed residual arrows ─────────────────────────────────────────────────
ax.annotate("", xy=(xi, yhat_i), xytext=(xi, yi),
            arrowprops=dict(arrowstyle="-", linestyle="dashed",
                            color="black", lw=1.2))
ax.annotate("", xy=(xj, yhat_j), xytext=(xj, yj),
            arrowprops=dict(arrowstyle="-", linestyle="dashed",
                            color="black", lw=1.2))

# ── 7. Labels: i, j, e_i, e_j ────────────────────────────────────────────────
ax.text(xi + 0.15, yi - 0.3,  "i",   fontsize=12, style="italic")
ax.text(xj - 0.4, yj + 0.0,  "j",   fontsize=12, style="italic")
ax.text(xi + 0.2,  (yi + yhat_i) / 2, "$e_i$", fontsize=12, style="italic")
ax.text(xj - 0.6,  (yj + yhat_j) / 2, "$e_j$", fontsize=12, style="italic")

# ── 8. Grid ───────────────────────────────────────────────────────────────────
ax.grid(True, linestyle="-", linewidth=0.4, color="lightgray", zorder=0)

# ── 9. Axis labels, limits, ticks ────────────────────────────────────────────
ax.set_xlabel("salt", fontsize=13)
ax.set_ylabel("BP", fontsize=13)
ax.set_xlim(-0.5, 14.5)
ax.set_ylim(127, 146)
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax.yaxis.set_major_locator(ticker.MultipleLocator(5))

ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
ax.tick_params(axis="both", direction="out", which="both",
               top=False, right=False, labelsize=11)

plt.tight_layout()
plt.show()

If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_11/chapter_11A_image13A.png)


The least-squares regression line for the relationship between blood pressure and sodium chloride intake. The vertical arrows show the residuals for two observations. The stars are the estimated blood pressure for daily sodium chloride
intakes from $0$ to $14$ grams.

As before, the regression line is specified by its intercept $a$ and its slope $b$ and can be used to estimate the value of the response variable. Given $a$ and $b$, we can obtain the point estimate $\hat{y}{i}$ for the value of the response variable for individual $i$, whose value of the explanatory variable is $x{i}$,
$$
\hat{y}_{i} = a + bx _{i}.
$$
In general, this estimated value is different from the observed value $y_{i}$ of the response variable for the individual $i$. We use $e_{i}$ to denote the residual, which is the difference between the estimated and observed value of the response variable,
$$
e_{i} = y_{i} - \hat{y}_{i}.
$$
The right panel of Fig. 11.7 shows the residuals (vertical arrows) for two observations in the blood pressure data. For a sample of n individuals, the discrepancy between the linear regression model and the observed data is measured using the residual sum of squares,
$$
RSS = \sum_{i}^{n} e_{i}^{2}.
\tag{11.4}
$$
As mentioned above, the least-squares regression line provides the smallest possible value of RSS among all candidate straight lines.

For each individual in our sample, we can write the observed value of the response variable in terms of the estimated value according to the linear regression model and the corresponding residual as follows:
$$
\begin{array}{l}
y_{i} = \hat {y}_{i} + e_{i} \\
= a + bx_{i} + e_{i}. \\
\end{array}
$$

Here, $a$ and $b$ are the point estimates for $\alpha$ and $\beta$, which are the parameters of the linear regression model for the entire population,
$$
Y = \alpha + \beta X + \varepsilon.
$$
For simple linear regression model with binary explanatory variables, we found $a$ and $b$ quite simply by using the sample means for the two groups. For simple linear regression models with numerical explanatory variables, we find $a$ and $b$ as follows.

---------------

>First, we find the slope of regression line using the sample correlation coefficient $r$ between the response variable $Y$ and the explanatory variable $X$,
$$b = r\frac {s_{Y}}{s_{X}}. $$
Here, $ s_{y} $ is the sample standard deviation of $Y$, and $s_{x}$ is the sample standard deviation of $X$. Note that since $s_{x}$ and $s_{y}$ are always positive, the sign of $b$ is the same as the sign of the correlation coefficient: $b > 0$ for positively correlated random variables, and $b < 0$ for negatively correlated variables. When $r = 0$ (i.e., the two variables are not linearly related), then $b = 0$. After finding the slope, we find the intercept as follows:
$$ a = \bar {y} - b \bar {x}, $$
where $\bar{y}$ and $\bar{x}$ are the sample means for $Y$ and $X$, respectively. Then the least-squares regression line with intercept $a$ and slope $b$ can be expressed as
$$ \hat {y} = a + b x. $$

--------

For the blood pressure example, the sample correlation coefficient is $r=0.84$; the sample standard deviation of blood pressure is $s_{y}=4.94$, and the sample standard deviation of sodium chloride intake is $s_{x}=3.46$. Therefore,
$$ b = 0. 8 4 \times \frac {4 . 9 4}{3 . 4 6} = 1. 2 0. $$
For the observed data, the sample means are $\bar{y}=135.68$ and $\bar{x}=5.90$. Therefore,
$$ a = 1 3 5. 6 8 - 1. 2 0 \times 5. 9 0 = 1 2 8. 6 0. $$
The linear regression model can be written as
$$ \hat {y} = 1 2 8. 6 0 + 1. 2 0 x. $$
We can now use this model to estimate the value of the response variable. For the individual $i$ in the right panel of Fig. 11.7, the amount of daily sodium chloride intake is $x_{i}=3.68$. The estimated value of the blood pressure for this person is
$$ \hat {y} _ {i} = 1 2 8. 6 0 + 1. 2 0 \times 3. 6 8 = 1 3 3. 0 2. $$

The actual blood pressure for this individual is $ y_{i}=128.3 $ . The residual therefore is
$$ e _ {i} = y _ {i} - \hat {y} _ {i} = 1 2 8. 3 - 1 3 3. 0 2 = - 4. 7 2. $$
In the right panel of Fig. 11.7, the arrow that represents the residual for observation $i$ starts from 133.02 (i.e., estimated value according to the regression model) and ends at 128.3 (i.e., the observed value of the blood pressure variable), and its length is 4.72; the negative sign corresponds to a downward arrow.

We can also use our model for predicting the unknown values of the response variable (i.e., blood pressure) for all individuals in the target population. For example, if we know the amount of daily sodium chloride intake is $x = 7.81$ for an individual, we can predict her blood pressure as follows:

$$
\hat {y} = 128.60 + 1.20 \times 7.81 = 137.97.
$$
Of course, the actual value of the blood pressure for this individual would be different from the predicted value. The difference between the actual and predicted values of the response variable is called the model **error** and is denoted as $\varepsilon$. In fact, the residuals are the observed values of $\varepsilon$ for the individuals in our sample.

In the right panel of Fig. 11.7, stars show the predicted values of the response variable for values of the explanatory variable from 0 to 14. These are the expected blood pressure values for people in the population of interest with 0, 1, 2, ..., 14 grams of daily sodium chloride intake. Note that in general, we should be cautious about using a regression model for prediction outside the population from which the sample is obtained.

For this example, we obtained the data from the population of the elderly people (above 65 years old). Using our model for predicting blood pressure of young people based on their daily amount of sodium chloride consumption would not be appropriate since the relationship between blood pressure and sodium chloride intake might not be the same among the young population. For example, the relationship might be much weaker.

We should also be cautious about using the regression line for prediction outside the range of observed values of the explanatory variables. For the above example, the observed values of $X$ in our sample are between 1 and 13. Using the regression line for prediction outside of this range is called **extrapolation**. As we move away from this range, the overall relationship between the response variable and the explanatory variable may change. In general, extrapolation far beyond the range of observed values for the explanatory variable is not recommended.

The predicted value for an individual with $x = 0$ (i.e., zero gram sodium chloride intake) is equal to the intercept:

$$
\hat {y}_{i} = 128.60 + 1.20 \times 0 = 128.60 = a.
$$
This is the point where the regression line intercepts the vertical axis. Therefore, the intercept is interpreted as the expected value of the blood pressure among people with zero sodium chloride diet. Of course, setting the value of the explanatory variable to zero does not always make sense. For example, if we use the weight of individuals as the explanatory variable for blood pressure, we cannot interpret the intercept as the expected value of the blood pressure among people whose weight is zero.

The interpretation of slope $b$ for the above model is similar to the interpretation of slope for simple linear regression models with a binary explanatory variable: the slope represents the expected (average) amount of change in the response variable for one unit increase in the value of the explanatory variable. For binary variables, one unit increase in $X$ meant moving from the group with $X = 0$ to the group with $X = 1$. For a numerical variable such as daily sodium chloride intake, one unit increase could mean increasing the daily amount of sodium chloride intake from $6$ to $7$ or from $10$ to $11$. For the above model, people whose daily sodium chloride intake is $7$ grams are expected (i.e., on average) to have $b = 1.20$ units higher blood pressure compared to those whose daily intake is $6$ grams. The same comment can be made for comparing people with $11$ grams of daily intake to those with $10$ grams of daily intake.

Finding confidence intervals for regression parameters $\alpha$ and $\beta$ also remains as before. More specifically, the confidence interval for regression coefficient is obtained as follows:

$$
[b - t_{crit} \times SE_{b}, b + t_{crit} \times S E _ {b} ].
$$
Here, $SE_{b}$ is the standard error of the regression coefficient and is calculated according to Eq. 11.3. We obtain $t_{crit}$ from the $t$-distribution with $n-2$ degrees of freedom for the given confidence level.

The steps for performing hypothesis testing regarding the linear relationship between the response and explanatory variables also remain the same. The null hypothesis is $H_{0}:\beta=0 $, which indicates that the two variables are not linearly related. This corresponds to a horizontal regression line, whose slope is zero. To evaluate this hypothesis, we need to find the $t$-score first,
$$
t = \frac {b}{SE_{b}}.
$$

Then, we find the $p$-value (i.e., the observed significance level) by calculating the probability of as or more extreme values than the t-score under the null hypothesis, in the direction of the alternative hypothesis. To this end, we use the $t$-distribution with $n-2$ degrees of freedom to find the lower tail probability of the $t$-score if $H_{A}:\beta<0 $, or its upper tail probability if $H_{A}:\beta>0 $, or two times the upper tail probability of its absolute value if $ H_{A}:\beta\neq0$.

The steps for using Python to find the points estimates and confidence intervals of regression parameters, and performing hypothesis testing based on linear regression models were demonstrated earlier in this lesson (Example 1).

# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) Exercises.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/STA1403/backend_code/validate.py?v=" + str(time.time())

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)

with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))

main()